In [1]:
import numpy as np
import time

n = 10_000_000
a = np.random.rand(n)
b = np.random.rand(n)

# Abordagem 1: Loop Python (interpretado, sequencial)
start = time.time()
c = [a[i] * b[i] for i in range(n)]
print(f"Loop Python:  {time.time() - start:.3f}s")

# Abordagem 2: Vetorizado NumPy (SIMD em C, paralelo)
start = time.time()
c = a * b
print(f"NumPy vetor:  {time.time() - start:.3f}s")

Loop Python:  1.358s
NumPy vetor:  0.093s


In [4]:
from numba import njit
import numpy as np
import time

@njit
def mult_loop(a, b):
    c = np.empty(len(a))
    for i in range(len(a)):
        c[i] = a[i] * b[i]
    return c

n = 10_000_000
a = np.random.rand(n)
b = np.random.rand(n)

# 1ª chamada: inclui compilação JIT
_ = mult_loop(a, b)

# 2ª chamada: já compilado
start = time.time()
c = mult_loop(a, b)
print(f"Numba JIT: {time.time()-start:.4f}s")

Numba JIT: 0.0277s


In [8]:
import numpy as np
from numba import njit
import time

# 1. Função em Python Puro (Péssima performance)
def logic_python(arr):
    n = len(arr)
    res = np.zeros(n)
    for i in range(1, n):
        # Uma lógica condicional complexa que depende do vizinho
        if arr[i] > 0.5:
            res[i] = np.sqrt(arr[i]) + res[i-1]
        else:
            res[i] = arr[i]**2 - res[i-1]
    return res

# 2. Tentativa com NumPy (Vetorização parcial/difícil)
# Nota: Para lógicas onde o i depende do i-1, o NumPy costuma exigir 
# o uso de 'np.ufunc.accumulate', que é bem menos intuitivo.
def logic_numpy(arr):
    # O NumPy sofre aqui porque não consegue "olhar para trás" 
    # de forma nativa e eficiente sem loops em C.
    return np.cumsum(np.where(arr > 0.5, np.sqrt(arr), arr**2)) # Aproximação simplificada

# 3. Numba (O brilho: loop nativo e extremamente rápido)
@njit
def logic_numba(arr):
    n = len(arr)
    res = np.empty(n)
    res[0] = arr[0]
    for i in range(1, n):
        if arr[i] > 0.5:
            res[i] = np.sqrt(arr[i]) + res[i-1]
        else:
            res[i] = arr[i]**2 - res[i-1]
    return res

# --- Teste de Performance ---
n = 10_000_000
data = np.random.rand(n)

print(f"Processando {n} elementos...\n")

# Warm-up Numba (Compilação)
_ = logic_numba(data)

# Timing Numba
start = time.time()
res_numba = logic_numba(data)
print(f"Numba: {time.time()-start:.6f}s")

# Timing Python Puro (limitado a menos elementos para não travar)
n_small = 100_000
start = time.time()
_ = logic_python(data[:n_small])
print(f"Python Puro (escalado p/ {n}): ~{(time.time()-start)*(n/n_small):.2f}s")

# Timing NumPy (se tentarmos fazer o loop manual no NumPy sem vetorizar)
start = time.time()
_ = logic_python(data) # Rodando o loop manual no array numpy
print(f"NumPy com loop manual: {time.time()-start:.6f}s")

Processando 10000000 elementos...

Numba: 0.076380s
Python Puro (escalado p/ 10000000): ~4.68s
NumPy com loop manual: 4.397333s


In [7]:
import cupy as cp
import numpy as np
import time

n = 10_000_000

# Dados na CPU (host)
a_cpu = np.random.rand(n)
b_cpu = np.random.rand(n)

# Transferir para GPU (device)
a_gpu = cp.asarray(a_cpu)
b_gpu = cp.asarray(b_cpu)

# Sincronizar antes de medir
cp.cuda.Stream.null.synchronize()

start = time.time()
c_gpu = a_gpu * b_gpu
cp.cuda.Stream.null.synchronize()
print(f"CuPy GPU: {time.time()-start:.5f}s")

# Trazer resultado de volta
c_cpu = cp.asnumpy(c_gpu)

CuPy GPU: 0.27612s


In [9]:
import hashlib
from multiprocessing import Pool
import time

def hash_data(data):
    return hashlib.sha256(
        data.encode()
    ).hexdigest()

# Gerar inputs
inputs = [f"dado_{i}" for i in range(100_000)]

# Sequencial (CPU single-thread)
start = time.time()
results_seq = [hash_data(d) for d in inputs]
print(f"Sequencial: {time.time()-start:.3f}s")

# Paralelo (CPU multi-core)
start = time.time()
with Pool() as pool:
    results_par = pool.map(hash_data, inputs)
print(f"Multiprocessing: {time.time()-start:.3f}s")

Sequencial: 0.048s
Multiprocessing: 0.192s
